# КИМ 7.1. Оптимизаторы и управление скоростью обучения

**Модуль 7. Алгоритмы обучения нейронных сетей** · Курс «Основы нейронных сетей» · УрФУ

Проверяемый результат (КРМ v3.0):
- **DL-1.1 (С, составной вклад M7):** выбирает скорость обучения под задачу,
  датасет и архитектуру и объясняет её связь с градиентным спуском.

M7 не подтверждает полный DL-1.1 С самостоятельно. Loss, `batch_size`,
регуляризация и Dropout проверяются шлюзом M2. Полный уровень возможен только при
одновременном выполнении `шлюз M2 = 1` и `шлюз M7 = 1`.

**Фиксированный стек КИМ:** PyTorch + Fashion-MNIST + MLP
`784 → 256 (ReLU, Dropout 0.2) → 10`.

Подробное описание, критерии и контрольные вопросы — в `kim-01-optimizers.md`.

---
## Часть А. Сравнение оптимизаторов

Используйте PyTorch, Fashion-MNIST и заданную MLP. Во всех запусках зафиксируйте
разбиение 20 000 train / 5 000 validation, `batch_size`, `CrossEntropyLoss`,
`Dropout(0.2)` и бюджет эпох. Внутри каждого seed синхронизируйте инициализацию,
порядок батчей и Dropout. В основной optimizer/lr-сетке для всех алгоритмов
установите `weight_decay=0`: регуляризацию подтверждает M2, а не M7.

До запуска запишите критерий выбора по validation. Не вычисляйте test-метрику,
пока итоговая конфигурация не зафиксирована.

### Критерий выбора до запуска

Запишите основную validation-метрику и правило разрешения равенства. Отдельно
задайте порог сходимости. Если порог не достигнут, храните и показывайте
`not reached`: не подменяйте его фиктивной эпохой и не используйте такую эпоху в
tie-break. Test в правило не входит.

> *Ваш заранее зафиксированный критерий:* ...

### 0. Импорт библиотек

In [ ]:
# < ENTER YOUR CODE HERE >

### 1. Fashion-MNIST и фиксированная подвыборка

In [ ]:
# < ENTER YOUR CODE HERE >  # PyTorch; 20_000 train + 5_000 validation; test пока не создавать

### 2. Фиксация архитектуры

Реализуйте PyTorch-модель `Flatten → Linear(784, 256) → ReLU → Dropout(0.2) →
Linear(256, 10)`. Последний слой возвращает логиты для `CrossEntropyLoss`.
Реализуйте `set_seed()` и `fresh_model(seed)`.

In [ ]:
# < ENTER YOUR CODE HERE >  # nn.Module + fresh_model(seed)

### 2.1. Базовый шаг градиентного спуска

Объясните смысл каждого элемента обновления
`theta_(t+1) = theta_t - lr * grad_theta L(theta_t)`. Свяжите величину `lr` со
скоростью и устойчивостью сходимости, включая риск перескоков при большом шаге и
медленного продвижения при малом.

> *Ваше объяснение:* ...

### 3. Fair-сетка optimizer × initial lr

Для **каждого** из `SGD`, `SGD(momentum=0.9)`, `Adam`, `AdamW`, `RMSProp`
задайте сетку одинакового размера минимум из трёх подходящих initial `lr`.
Выполните не менее 5×3 запусков с `weight_decay=0`. Выберите лучший `lr` каждого
optimizer по validation. Не выбирайте optimizer до настройки всех пяти.

In [ ]:
# < ENTER YOUR CODE HERE >  # 5 optimizers x >= 3 lr; weight_decay=0

### 4. Сравнение настроенных оптимизаторов

Покажите полную таблицу 5×3 и кривые большого/малого `lr` для каждого optimizer.
Затем сравните на сводных графиках пять пар с validation-выбранными `lr`.
Convergence показывайте как эпоху или `not reached`. Объясните, что AdamW при
`weight_decay=0` включён как алгоритм, но этот эксперимент не проверяет пользу
decoupled weight decay. Test не просматривайте.

In [ ]:
# < ENTER YOUR CODE HERE >  # таблица 5x3 + пять tuned-кривых + объяснение

**Контрольный вопрос:** чем SGD отличается от Adam и RMSProp? Что такое
момент (momentum) и адаптивная скорость обучения? Почему Adam сходится быстрее
SGD на практике?

> *Ваш ответ:* ...

---
## Часть Б. Управление скоростью обучения (lr scheduling)

### 5. Стратегия изменения lr

Примените `ReduceLROnPlateau(factor=0.5, patience=5)` и/или косинусное затухание
(`CosineAnnealingLR`). Для лучшей tuned-пары при одинаковом бюджете сравните
scheduler с **выбранным тем же
постоянным initial `lr`** и покажите траекторию `lr`. Объясните, почему scheduler
не заменяет выбор initial `lr`.

In [ ]:
# < ENTER YOUR CODE HERE >  # lr scheduler

---
## Часть В. Ранняя остановка

### 6. Ранняя остановка

До запуска письменно обоснуйте `monitor='val_accuracy'` и `patience=15`.
Реализуйте early stopping с восстановлением best state. Явно выведите, сработала
ли остановка и сколько эпох сэкономлено. Отсутствие остановки до лимита допустимо;
проверка корректного восстановления обязательна в обоих случаях.

> *Обоснование monitor/patience до запуска:* ...

In [ ]:
# < ENTER YOUR CODE HERE >  # EarlyStopping

**Контрольный вопрос:** как работает `ReduceLROnPlateau`? Зачем
уменьшать lr по ходу обучения? Что делает `EarlyStopping` и зачем
`restore_best_weights=True`?

> *Ваш ответ:* ...

---
## Часть Г. Устойчивость и окончательное решение

### 7. Проверка на seed

По validation выберите предварительно лучшую полную конфигурацию
`optimizer + initial lr + scheduler` и ближайшую альтернативу. Повторите только
эти два варианта минимум на трёх seed. Покажите для каждого mean/std
validation-метрики, долю достижения convergence-порога и среднюю эпоху только
среди достигших; всю сетку повторять не требуется.

In [ ]:
# < ENTER YOUR CODE HERE >  # 2 конфигурации x >= 3 seed; mean/std + success rate

### 8. Финальный выбор и test

До test примените прозрачное правило: кандидаты в пределах `0.002` от лучшего
validation mean, затем минимальный std, затем большая доля достижения порога и
меньшая средняя эпоха среди достигших. Зафиксируйте итоговую конфигурацию. Только
теперь создайте test loader и один раз оцените test. Составьте decision table:
`optimizer`, `initial lr`, `scheduler`, `validation metric`, `convergence`,
`seed stability`.

In [ ]:
# < ENTER YOUR CODE HERE >  # decision table + единственная финальная test-оценка

---
## 9. Вывод

Какой `optimizer + initial lr + scheduler` выбран и почему? Дайте рекомендации,
явно ограничив их Fashion-MNIST, данной MLP и протоколом. Отдельно
укажите, почему `шлюз M7 = 1` не является самостоятельным полным подтверждением:
полный DL-1.1 С требует `шлюз M2 = 1` и `шлюз M7 = 1`.

> *Ваш вывод:* ...

---
## Итог

Перед сдачей убедитесь, что:
- [ ] выполнена полная сетка 5 optimizers × минимум 3 lr с `weight_decay=0`;
- [ ] лучший lr каждого optimizer выбран по validation до сравнения optimizer;
- [ ] есть таблица 5×3, tuned-кривые и объяснение большого/малого lr;
- [ ] scheduler сравнен с выбранным постоянным lr и не назван его заменой;
- [ ] `not reached` не заменён фиктивной эпохой;
- [ ] две конфигурации повторены на трёх seed: mean/std и success rate;
- [ ] monitor/patience обоснованы, best state проверен даже без остановки;
- [ ] правило допуска 0.002 и выбора по std зафиксировано до test;
- [ ] decision table содержит optimizer/lr/scheduler/validation/convergence/stability;
- [ ] EarlyStopping применён, статус и восстановление best state показаны;
- [ ] test просмотрен только после выбора итоговой конфигурации;
- [ ] вы готовы объяснить базовый шаг GD и механизм ReduceLROnPlateau.

**Формат сдачи:** заполненный ноутбук + устная защита.